# 01 — Data Loading

This notebook loads and inspects the two core datasets from the [DepMap portal](https://depmap.org/portal/) that power this project.

> **Files were downloaded manually** from [depmap.org/portal/data_page](https://depmap.org/portal/data_page/) and placed in `data/raw/`. See the manual download instructions at the bottom of this notebook if you need to re-download them.

---

## Why CCLE / DepMap?

The **Cancer Cell Line Encyclopedia (CCLE)**, distributed through the Broad Institute's DepMap portal, is the primary data source for this project for several reasons:

| Reason | Detail |
|--------|--------|
| **Scale** | ~1,800+ cancer cell lines across >30 tissue lineages |
| **Data alignment** | RNA-seq expression and mutation calls are derived from the *same* cell line cultures — no cross-cohort batch effects |
| **Curation quality** | Mutations are called with a standardised pipeline (GATK/MuTect2), then filtered for germline artefacts and manually reviewed for key cancer genes |
| **Public access** | Freely available; no data-use agreement required for the files used here |
| **TP53 coverage** | TP53 is one of the most frequently mutated genes in CCLE (~40–60% of lines depending on lineage), giving sufficient positive examples for classification |

---

## Files we use

### 1. `OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv` — gene expression

- **What it is**: RNA-seq expression values for all human protein-coding genes, normalised to **log₂(TPM + 1)**. The `+1` pseudo-count prevents log(0) and compresses the dynamic range of lowly expressed genes.
- **Shape**: `(cell lines) × (genes)` — roughly 1,450 rows × 19,000 columns.
- **Index column**: `DepMap_ID` — a stable identifier like `ACH-000001`.
- **Gene columns**: formatted as `SYMBOL (ENTREZ_ID)`, e.g. `TP53 (7157)`. We will strip the Entrez suffix during preprocessing.
- **This is our feature matrix X** for every downstream model.

### 2. `OmicsSomaticMutations.csv` — somatic mutation calls

- **What it is**: One row per mutation event (variant-level, not cell-line-level). Aggregating by `ModelID` + `HugoSymbol == 'TP53'` gives us our target labels.
- **Key columns we expect**:

| Column | Description |
|--------|-------------|
| `ModelID` | DepMap cell line identifier (= `DepMap_ID`) |
| `HugoSymbol` | Gene name (we filter for `TP53`) |
| `VariantClassification` | Mutation consequence: `Missense_Mutation`, `Nonsense_Mutation`, `Frame_Shift_Del/Ins`, `Splice_Site`, etc. |
| `VariantType` | `SNP`, `DEL`, `INS` |
| `Chromosome`, `Start_Pos`, `End_Pos` | Genomic coordinates |
| `ReferenceAllele`, `AltAllele` | Alleles |
| `ProteinChange` | HGVS protein-level notation, e.g. `p.R175H` |
| `isCOSMIChotspot` | Boolean — whether this variant is a COSMIC hotspot |

- From this file we will derive **binary labels** (TP53 mutant vs. wild-type) and **multi-class labels** (missense / nonsense / frameshift / splice / WT) for Tasks 1 and 2.

---

## Setup

In [17]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")

EXPR_FILE = RAW_DIR / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
MUT_FILE  = RAW_DIR / "OmicsSomaticMutations.csv"

for f in (EXPR_FILE, MUT_FILE):
    status = "OK" if f.exists() else "MISSING"
    print(f"[{status}] {f.name}")

[OK] OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv
[OK] OmicsSomaticMutations.csv


---

## Inspect: Gene Expression

In [18]:
expr = pd.read_csv(EXPR_FILE, index_col=0)

print("Shape:", expr.shape)
print(f"  {expr.shape[0]:,} cell lines × {expr.shape[1]:,} genes")
print()
print("Index name:", expr.index.name)
print("First 5 index values:", expr.index[:5].tolist())
print()
print("First 5 gene column names:", expr.columns[:5].tolist())

Shape: (1775, 19220)
  1,775 cell lines × 19,220 genes

Index name: None
First 5 index values: [0, 1, 2, 3, 4]

First 5 gene column names: ['SequencingID', 'ModelConditionID', 'ModelID', 'IsDefaultEntryForMC', 'IsDefaultEntryForModel']


In [19]:
expr.iloc[:5, :8]

,SequencingID,ModelConditionID,ModelID,IsDefaultEntryForMC,IsDefaultEntryForModel,TSPAN6 (7105),TNMD (64102),DPM1 (8813)
0,CDS-010xbm,MC-001113-k2lR,ACH-001113,Yes,Yes,4.956577,0.000000,7.577648
1,CDS-02TzJp,MC-001289-BpdI,ACH-001289,Yes,Yes,4.955015,0.617117,7.333933
2,CDS-0693hw,MC-001339-5nRN,ACH-001339,Yes,Yes,3.421952,0.000000,7.546069
3,CDS-07Plat,MC-001619-IR6I,ACH-001619,No,No,5.196729,0.000000,6.362268
4,CDS-08FOcu,MC-001979-E3qW,ACH-001979,Yes,Yes,4.651643,0.000000,5.946408


In [20]:
# Sanity-check value range — log2(TPM+1) should be in roughly [0, 20]
# Avoid stack() which can infer object dtype on mixed-type DataFrames
import numpy as np

numeric = expr.select_dtypes(include="number")
print(f"Numeric columns : {numeric.shape[1]:,} / {expr.shape[1]:,} total")
print()
flat = numeric.values.ravel().astype(float)
print(pd.Series(flat).describe())
print()
print(f"Values outside [0, 20]: {((flat < 0) | (flat > 20)).sum():,} "
      f"({((flat < 0) | (flat > 20)).mean() * 100:.2f}%)")

Numeric columns : 19,215 / 19,220 total

count    3.410662e+07
mean     2.670014e+00
std      2.548658e+00
min      0.000000e+00
25%      6.729452e-02
50%      2.415680e+00
75%      4.617933e+00
max      1.736064e+01
dtype: float64

Values outside [0, 20]: 0 (0.00%)


In [21]:
# Check that TP53 is present — column format varies by release:
#   'TP53 (7157)'  (symbol + Entrez ID)  or just  'TP53'
tp53_cols = [c for c in expr.columns if c == "TP53" or c.startswith("TP53 (")]
print("TP53 expression column(s):", tp53_cols)

if not tp53_cols:
    print("\nTP53 column not found in expected format.")
    print("Sample column names (first 10):")
    print(expr.columns[:10].tolist())
else:
    print()
    print(expr[tp53_cols[0]].describe())

TP53 expression column(s): ['TP53 (7157)']

count    1775.000000
mean        5.104906
std         1.380910
min         0.000000
25%         4.407970
50%         5.378661
75%         6.058212
max         8.101655
Name: TP53 (7157), dtype: float64


---

## Inspect: Somatic Mutations

In [22]:
mut = pd.read_csv(MUT_FILE, low_memory=False)

print("Shape:", mut.shape)
print()
print("Columns:")
print(mut.columns.tolist())

# Identify the cell line ID column — 'ModelID' in recent releases, 'DepMap_ID' in older ones
id_col = "ModelID" if "ModelID" in mut.columns else "DepMap_ID"
gene_col = "HugoSymbol" if "HugoSymbol" in mut.columns else "Gene"
print()
print(f"Using cell line ID column : '{id_col}'")
print(f"Using gene symbol column  : '{gene_col}'")
print(f"Mutation events           : {len(mut):,}")
print(f"Unique cell lines         : {mut[id_col].nunique():,}")

Shape: (1172688, 69)

Columns:
['Unnamed: 0', 'SequencingID', 'ModelID', 'ModelConditionID', 'IsDefaultEntryForModel', 'IsDefaultEntryForMC', 'Chrom', 'Pos', 'Ref', 'Alt', 'AF', 'DP', 'RefCount', 'AltCount', 'GT', 'PS', 'VariantType', 'VariantInfo', 'DNAChange', 'ProteinChange', 'HugoSymbol', 'Exon', 'Intron', 'EnsemblGeneID', 'EnsemblFeatureID', 'HgncName', 'HgncFamily', 'UniprotID', 'DbsnpRsID', 'GcContent', 'NMD', 'MolecularConsequence', 'VepImpact', 'VepBiotype', 'VepHgncID', 'VepExistingVariation', 'VepManeSelect', 'VepENSP', 'VepSwissprot', 'Sift', 'Polyphen', 'GnomadeAF', 'GnomadgAF', 'VepClinSig', 'VepSomatic', 'VepPliGeneValue', 'VepLofTool', 'OncogeneHighImpact', 'TumorSuppressorHighImpact', 'TranscriptLikelyLof', 'Brca1FuncScore', 'CivicID', 'CivicDescription', 'CivicScore', 'LikelyLoF', 'HessDriver', 'HessSignature', 'RevelScore', 'PharmgkbId', 'GwasDisease', 'GwasPmID', 'GtexGene', 'ProveanPrediction', 'AMClass', 'AMPathogenicity', 'Rescue', 'RescueReason', 'Hotspot', 'Ent

In [23]:
mut.head(5)

,Unnamed: 0,SequencingID,ModelID,ModelConditionID,IsDefaultEntryForModel,IsDefaultEntryForMC,Chrom,Pos,Ref,Alt,...,GwasDisease,GwasPmID,GtexGene,ProveanPrediction,AMClass,AMPathogenicity,Rescue,RescueReason,Hotspot,EntrezGeneID
0,0,CDS-aN8PNg,ACH-000062,MC-000062-eiDD,Yes,Yes,chr1,818203,G,A,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,400728.0
1,1,CDS-qjpRtt,ACH-002059,MC-002059-DNM1,Yes,Yes,chr1,918916,CTGA,C,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,100130417.0
2,2,CDS-bOBkBi,ACH-000402,MC-000402-iC8K,Yes,Yes,chr1,924510,GC,AA,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,148398.0
3,3,CDS-52aoJ0,ACH-000693,MC-000693-vUEr,Yes,Yes,chr1,924657,C,G,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,148398.0
4,4,CDS-jBuyoZ,ACH-000930,MC-000930-v0gO,Yes,Yes,chr1,924750,C,T,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,148398.0


In [24]:
#select the columns VariantType,VariantInfo,DNAChange,ProteinChange,HugoSymbol
mut[["VariantType", "VariantInfo", "DNAChange", "ProteinChange", "HugoSymbol"]].head(5)

#EDA of each of these columns
#We want to see the distribution of the variants types, DNA changes, protein changes, and hugo symbols
#If ProteinChange is NaN we want the relative VariantInfo to be grouped together, it's because it doesn't have a functional effect
#For not NaN ProteinChange, we want to see the distribution of the Variant Info

,VariantType,VariantInfo,DNAChange,ProteinChange,HugoSymbol
0,SNV,splice_donor_variant&non_coding_transcript_var...,ENST00000326734.3:n.840+1G>A,NaN,FAM87B
1,deletion,splice_acceptor_variant&splice_polypyrimidine_...,ENST00000756700.1:n.623-4_623-2del,NaN,LINC02593
2,substitution,missense_variant,ENST00000616016.5:c.79_80delinsAA,p.A27K,SAMD11
3,SNV,missense_variant,ENST00000616016.5:c.226C>G,p.L76V,SAMD11
4,SNV,missense_variant,ENST00000616016.5:c.319C>T,p.P107S,SAMD11


In [25]:
# TP53 mutations only
tp53_mut = mut[mut[gene_col] == "TP53"].copy()

print(f"TP53 mutation events            : {len(tp53_mut):,}")
print(f"Cell lines with TP53 mutation   : {tp53_mut[id_col].nunique():,}")

if "VariantClassification" in tp53_mut.columns:
    print()
    print("VariantClassification breakdown:")
    print(tp53_mut["VariantClassification"].value_counts())
else:
    print()
    print("'VariantClassification' column not found. Available columns:")
    print(tp53_mut.columns.tolist())

TP53 mutation events            : 2,138
Cell lines with TP53 mutation   : 1,189

'VariantClassification' column not found. Available columns:
['Unnamed: 0', 'SequencingID', 'ModelID', 'ModelConditionID', 'IsDefaultEntryForModel', 'IsDefaultEntryForMC', 'Chrom', 'Pos', 'Ref', 'Alt', 'AF', 'DP', 'RefCount', 'AltCount', 'GT', 'PS', 'VariantType', 'VariantInfo', 'DNAChange', 'ProteinChange', 'HugoSymbol', 'Exon', 'Intron', 'EnsemblGeneID', 'EnsemblFeatureID', 'HgncName', 'HgncFamily', 'UniprotID', 'DbsnpRsID', 'GcContent', 'NMD', 'MolecularConsequence', 'VepImpact', 'VepBiotype', 'VepHgncID', 'VepExistingVariation', 'VepManeSelect', 'VepENSP', 'VepSwissprot', 'Sift', 'Polyphen', 'GnomadeAF', 'GnomadgAF', 'VepClinSig', 'VepSomatic', 'VepPliGeneValue', 'VepLofTool', 'OncogeneHighImpact', 'TumorSuppressorHighImpact', 'TranscriptLikelyLof', 'Brca1FuncScore', 'CivicID', 'CivicDescription', 'CivicScore', 'LikelyLoF', 'HessDriver', 'HessSignature', 'RevelScore', 'PharmgkbId', 'GwasDisease', 'Gwas

In [26]:
# Select key columns that exist in this release
desired = [id_col, gene_col, "VariantClassification", "VariantType", "ProteinChange", "isCOSMIChotspot"]
available = [c for c in desired if c in tp53_mut.columns]
missing = [c for c in desired if c not in tp53_mut.columns]

if missing:
    print(f"Columns not found in this release (skipped): {missing}")

tp53_mut[available].head(10)

Columns not found in this release (skipped): ['VariantClassification', 'isCOSMIChotspot']


,ModelID,HugoSymbol,VariantType,ProteinChange
155226,ACH-000940,TP53,SNV,p.G389W
155227,ACH-000632,TP53,deletion,p.K382NfsTer40
155228,ACH-000997,TP53,SNV,NaN
155229,ACH-001061,TP53,SNV,NaN
155230,ACH-000227,TP53,deletion,NaN
155231,ACH-001975,TP53,substitution,p.L350P
155232,ACH-002283,TP53,SNV,p.L350P
155233,ACH-000720,TP53,SNV,p.E349Ter
155234,ACH-000701,TP53,SNV,p.A347V
155235,ACH-000803,TP53,SNV,p.A347P


---

## Summary

| File | Rows | Columns | Notes |
|------|------|---------|-------|
| Expression | ~1,450 (cell lines) | ~19,000 (genes) | log₂(TPM+1); index = `DepMap_ID` |
| Mutations | ~1.8M (events) | ~30+ | One row per variant; filter on `HugoSymbol == 'TP53'` for labels |

Both files are in `data/raw/`. The next notebook (`02_preprocessing.ipynb`) will:
- Align cell lines present in both datasets
- Derive binary and multi-class TP53 labels
- Handle class imbalance and split into train/test sets

---

## Manual download instructions

If you need to re-download these files:

1. Go to **[depmap.org/portal/data_page](https://depmap.org/portal/data_page/)**
2. Select the desired release from the **Release** dropdown.
3. Search for and download:
   - `OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv`
   - `OmicsSomaticMutations.csv`
4. Place both files in `data/raw/` and re-run this notebook.